In [ ]:
# Kaggle environment check – lists all input files
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


## Step 1 · Imports

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import matplotlib.cm as cm

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

print("TensorFlow:", tf.__version__)


## Step 2 · Load training data

In [ ]:
DATA_DIR  = "/kaggle/input/street-view-getting-started-with-julia"
WORK_DIR  = "/kaggle/working"

# Extract training images (skip if already extracted)
train_zip = os.path.join(DATA_DIR, "trainResized.zip")
if not os.path.isdir(os.path.join(WORK_DIR, "trainResized")):
    with zipfile.ZipFile(train_zip, "r") as z:
        z.extractall(WORK_DIR)

df = pd.read_csv(os.path.join(DATA_DIR, "trainLabels.csv"))
df = df[df["Class"].apply(lambda x: str(x).isdigit())].copy()
df["Class"] = df["Class"].astype(int)
df = df.reset_index(drop=True)

print(f"Training samples : {len(df)}")
print(f"Classes          : {sorted(df['Class'].unique())}")

# Class distribution
plt.figure(figsize=(8, 3))
sns.countplot(x="Class", data=df, palette="tab10")  # explicit palette avoids deprecation warning
plt.title("Class Distribution (train)")
plt.tight_layout()
plt.show()


## Step 3 · Load & resize images (64×64 RGB)

In [ ]:
IMG_SIZE = 64
X, y = [], []

for _, row in df.iterrows():
    path = os.path.join(WORK_DIR, "trainResized", f"{row['ID']}.Bmp")
    img  = cv2.imread(path)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # fix: BGR→RGB so colours are correct
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    X.append(img)
    y.append(row["Class"])

X = np.array(X, dtype=np.uint8)
y = np.array(y)

print(f"X shape : {X.shape}  |  y shape : {y.shape}")

# Show 10 random samples
idx = np.random.choice(len(X), 10, replace=False)
plt.figure(figsize=(12, 2))
for j, i in enumerate(idx):
    plt.subplot(1, 10, j + 1)
    plt.imshow(X[i])
    plt.title(y[i], fontsize=9)
    plt.axis("off")
plt.suptitle("Sample images (raw 64×64 RGB)", y=1.02)
plt.tight_layout()
plt.show()


## Step 4 · Preprocessing & train/val split

In [ ]:
# Cast to float32 before model-specific preprocessing
X_f = X.astype(np.float32)

X_m = mobilenet_preprocess(X_f.copy())   # scales to [-1, 1]
X_r = resnet_preprocess(X_f.copy())      # scales to ImageNet mean-subtracted

y_cat = to_categorical(y, num_classes=10)

X_train_m, X_val_m, y_train, y_val = train_test_split(
    X_m, y_cat, test_size=0.2, random_state=42, stratify=y)   # fix: stratify for balanced split
X_train_r, X_val_r, _, _            = train_test_split(
    X_r, y_cat, test_size=0.2, random_state=42, stratify=y)

print(f"Train : {X_train_m.shape[0]}  |  Val : {X_val_m.shape[0]}")

# Show MobileNet-preprocessed images (un-normalise for display)
plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    disp = ((X_train_m[i] + 1) * 127.5).clip(0, 255).astype(np.uint8)
    plt.imshow(disp)
    plt.axis("off")
plt.suptitle("MobileNetV2 preprocessed (display-normalised)", y=1.02)
plt.tight_layout()
plt.show()


## Step 5 · Data augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,   # digits must not be flipped
)
datagen.fit(X_train_m)

# Visualise 10 augmented variants of the first training image
sample = X_train_m[0:1]
plt.figure(figsize=(12, 2))
for i, batch in enumerate(datagen.flow(sample, batch_size=1, seed=0)):
    disp = ((batch[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
    plt.subplot(1, 10, i + 1)
    plt.imshow(disp)
    plt.axis("off")
    if i == 9:
        break
plt.suptitle("10 augmented variants of one training image", y=1.02)
plt.tight_layout()
plt.show()


## Step 6 · Build MobileNetV2 model

In [ ]:
def build_model(base_model):
    """Attach a classification head to a frozen backbone."""
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.5)(x)
    out = Dense(10, activation="softmax")(x)
    model = Model(inputs=base_model.input, outputs=out)
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, verbose=1),
    EarlyStopping(monitor="val_loss",   patience=5, restore_best_weights=True, verbose=1),
]

base_m  = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None)
model_m = build_model(base_m)
model_m.summary()


## Step 7 · Train MobileNetV2

In [ ]:
history_m = model_m.fit(
    datagen.flow(X_train_m, y_train, batch_size=64),
    validation_data=(X_val_m, y_val),
    epochs=30,
    callbacks=callbacks,
    verbose=2,
)


## Step 8 · Build & train ResNet50

In [ ]:
datagen_r = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,
)
datagen_r.fit(X_train_r)

base_r  = ResNet50(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None)
model_r = build_model(base_r)

history_r = model_r.fit(
    datagen_r.flow(X_train_r, y_train, batch_size=64),
    validation_data=(X_val_r, y_val),
    epochs=30,
    callbacks=callbacks,
    verbose=2,
)


## Step 9 · Training curves

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, metric, ylabel in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
        ax.plot(history.history[metric],     label="train")
        ax.plot(history.history[f"val_{metric}"], label="val",  linestyle="--")
        ax.set_title(f"{title} – {ylabel}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_history(history_m, "MobileNetV2")
plot_history(history_r, "ResNet50")


## Step 10 · Load & preprocess test images

In [ ]:
test_zip = os.path.join(DATA_DIR, "testResized.zip")
if not os.path.isdir(os.path.join(WORK_DIR, "testResized")):
    with zipfile.ZipFile(test_zip, "r") as z:
        z.extractall(WORK_DIR)

X_test, image_ids = [], []

for fname in sorted(os.listdir(os.path.join(WORK_DIR, "testResized"))):
    if not fname.lower().endswith(".bmp"):
        continue
    path = os.path.join(WORK_DIR, "testResized", fname)
    img  = cv2.imread(path)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # fix: BGR→RGB
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    X_test.append(img)
    image_ids.append(os.path.splitext(fname)[0])  # fix: robust extension stripping

X_test   = np.array(X_test, dtype=np.uint8)
X_test_m = mobilenet_preprocess(X_test.astype(np.float32).copy())
X_test_r = resnet_preprocess(X_test.astype(np.float32).copy())

print(f"Test samples : {len(X_test)}")

# Show first 10 test images
plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_test[i])
    plt.axis("off")
plt.suptitle("Sample test images", y=1.02)
plt.tight_layout()
plt.show()


## Step 11 · Ensemble prediction

In [ ]:
preds_m = model_m.predict(X_test_m, verbose=1)
preds_r = model_r.predict(X_test_r, verbose=1)

# Simple average ensemble – equal weight for both models
ensemble_preds = (preds_m + preds_r) / 2
final_preds    = np.argmax(ensemble_preds, axis=1)

# Show 10 predictions
plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_test[i])
    plt.title(str(final_preds[i]), fontsize=10)
    plt.axis("off")
plt.suptitle("Ensemble predictions on test images", y=1.02)
plt.tight_layout()
plt.show()


## Step 12 · Save submission

In [ ]:
submission = pd.DataFrame({"ID": image_ids, "Class": final_preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv written – shape:", submission.shape)
submission.head(10)


## Step 13 · Grad-CAM explainability

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Compute a Grad-CAM heatmap for the given image and model.

    Parameters
    ----------
    img_array         : np.ndarray, shape (1, H, W, 3), preprocessed
    model             : compiled Keras model
    last_conv_layer_name : str, name of the last convolutional layer
    pred_index        : int or None – class index to explain; defaults to top class

    Returns
    -------
    heatmap : np.ndarray, shape (h, w), values in [0, 1]
    """
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )

    img_tensor = tf.cast(img_array, tf.float32)   # fix: explicit float32 cast for GradientTape

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_outputs, predictions = grad_model(img_tensor)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads        = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap  = tf.squeeze(heatmap)
    heatmap  = tf.maximum(heatmap, 0)

    max_val = tf.math.reduce_max(heatmap)
    if max_val > 0:                                # fix: avoid division by zero
        heatmap = heatmap / max_val
    return heatmap.numpy()


In [ ]:
def overlay_heatmap(img_rgb, heatmap, alpha=0.4, cmap="jet"):
    """Blend a Grad-CAM heatmap onto an RGB image."""
    heatmap_resized  = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_colored  = plt.get_cmap(cmap)(heatmap_resized)[..., :3]   # fix: plt.get_cmap (cm.get_cmap deprecated)
    heatmap_colored  = (heatmap_colored * 255).astype(np.uint8)
    overlay          = cv2.addWeighted(img_rgb, 1 - alpha, heatmap_colored, alpha, 0)
    return overlay


In [ ]:
GRADCAM_INDICES = [0, 5, 10, 15, 20]

for idx in GRADCAM_INDICES:
    img_raw = X_test[idx]
    img_m   = X_test_m[idx:idx+1]
    img_r   = X_test_r[idx:idx+1]

    heatmap_m = make_gradcam_heatmap(img_m, model_m, last_conv_layer_name="Conv_1")
    heatmap_r = make_gradcam_heatmap(img_r, model_r, last_conv_layer_name="conv5_block3_out")

    overlay_m = overlay_heatmap(img_raw, heatmap_m)
    overlay_r = overlay_heatmap(img_raw, heatmap_r)

    fig, axes = plt.subplots(1, 3, figsize=(7, 2))
    for ax, img, title in zip(
        axes,
        [img_raw, overlay_m, overlay_r],
        ["Original", "MobileNetV2", "ResNet50"],
    ):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(f"Grad-CAM · test image index {idx}  →  pred: {final_preds[idx]}")
    plt.tight_layout()
    plt.show()
